# 6a Compare Reviews Original

This notebook runs the exact-n matched review-diversity analyses on the original-text review branch.

Within each condition it produces both:
- pooled Human vs All-AI review-panel comparisons
- per-model Human vs Claude/Gemini/GPT comparisons

It reuses the prepared outputs from `4b_prepare_review_for_analysis.ipynb`, saves reusable exact-panel caches, writes condition-specific tables/figures, and then exports cross-condition summaries.


In [ ]:
CONDITIONS_TO_RUN = ['baseline', 'one_at_a_time', 'persona']
TEXT_VERSION = 'original'
COMPARISONS = ['all_ai', 'claude', 'gemini', 'gpt']
CONFIRMATORY_METRICS = ['mean_pairwise', 'nn']
EXPLORATORY_METRICS = [
    'centroid_loo',
    'global_centroid_dist',
    'medoid_dist',
    'span90',
    'mst_dispersion',
    'sparseness',
]
COMPATIBILITY_METRICS = ['remote_clique']
ALL_METRICS = CONFIRMATORY_METRICS + EXPLORATORY_METRICS + COMPATIBILITY_METRICS
MIN_HUMAN_PANEL_FOR_SENSITIVITY = 3
REUSE_COMBINATION_CACHE = True


In [ ]:
import itertools
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import wilcoxon

import sys
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from compare_review_diversity import (
    COMPARISON_LABELS,
    build_exact_n_panel_combinations,
    build_review_diversity_proposal_master,
    compute_review_metric_correlation_table,
    cross_condition_summary_table,
    load_review_analysis_inputs,
    paired_review_diversity_tests,
    plot_review_diversity_effects,
    plot_review_diversity_paired_slopes,
    plot_review_embedding_space,
    save_pickle,
)
from proposal_generation import find_project_root

PROJECT_ROOT = find_project_root(PROJECT_ROOT)
sns.set_theme(style='whitegrid', context='talk')

def cross_condition_ai_contrast_table(proposal_master_df):
    rows = []
    conditions = sorted(proposal_master_df['condition'].dropna().unique().tolist())
    for comparison in sorted(proposal_master_df['comparison'].dropna().unique().tolist()):
        sub = proposal_master_df[proposal_master_df['comparison'] == comparison].copy()
        for metric in sorted(sub['metric'].dropna().unique().tolist()):
            metric_df = sub[sub['metric'] == metric][['condition', 'target_proposal_uid', 'ai_metric_panel_mean']].dropna()
            for left, right in itertools.combinations(conditions, 2):
                left_df = metric_df[metric_df['condition'] == left][['target_proposal_uid', 'ai_metric_panel_mean']].rename(columns={'ai_metric_panel_mean': 'left_value'})
                right_df = metric_df[metric_df['condition'] == right][['target_proposal_uid', 'ai_metric_panel_mean']].rename(columns={'ai_metric_panel_mean': 'right_value'})
                merged = left_df.merge(right_df, on='target_proposal_uid', how='inner')
                if merged.empty:
                    continue
                diff = merged['left_value'] - merged['right_value']
                if np.allclose(diff, 0):
                    stat = 0.0
                    p_value = 1.0
                else:
                    test = wilcoxon(merged['left_value'], merged['right_value'], zero_method='wilcox', alternative='two-sided', mode='auto')
                    stat = float(test.statistic)
                    p_value = float(test.pvalue)
                rows.append({
                    'comparison': comparison,
                    'comparison_label': COMPARISON_LABELS.get(comparison, comparison),
                    'metric': metric,
                    'left_condition': left,
                    'right_condition': right,
                    'n_proposals': int(len(merged)),
                    'left_mean_ai_metric': float(merged['left_value'].mean()),
                    'right_mean_ai_metric': float(merged['right_value'].mean()),
                    'mean_difference_left_minus_right': float(diff.mean()),
                    'median_difference_left_minus_right': float(diff.median()),
                    'wilcoxon_statistic': stat,
                    'p_value': p_value,
                })
    return pd.DataFrame(rows)

print(f'Project root: {PROJECT_ROOT}')
print(f'Conditions: {CONDITIONS_TO_RUN}')


In [ ]:
condition_outputs = {}
cross_condition_test_frames = []
cross_condition_proposal_frames = []

for condition in CONDITIONS_TO_RUN:
    print(f'\n=== 6a original review comparison: {condition} ===')
    analysis = load_review_analysis_inputs(PROJECT_ROOT, condition, text_version=TEXT_VERSION)

    if analysis.panel_registry['target_proposal_uid'].nunique() != 23:
        raise RuntimeError(f'{condition}: expected 23 target proposals, found {analysis.panel_registry["target_proposal_uid"].nunique()}')

    tables_all_ai_dir = PROJECT_ROOT / 'results' / 'tables' / condition / 'reviews' / TEXT_VERSION / 'all_ai'
    figures_all_ai_dir = PROJECT_ROOT / 'results' / 'figures' / condition / 'reviews' / TEXT_VERSION / 'all_ai'
    tables_per_model_dir = PROJECT_ROOT / 'results' / 'tables' / condition / 'reviews' / TEXT_VERSION / 'per_model'
    figures_per_model_dir = PROJECT_ROOT / 'results' / 'figures' / condition / 'reviews' / TEXT_VERSION / 'per_model'
    shared_cache_dir = PROJECT_ROOT / 'results' / 'tables' / condition / 'reviews' / TEXT_VERSION / 'shared_cache'
    for path in [tables_all_ai_dir, figures_all_ai_dir, tables_per_model_dir, figures_per_model_dir, shared_cache_dir]:
        path.mkdir(parents=True, exist_ok=True)

    combination_cache_path = shared_cache_dir / 'exact_n_panel_combinations.pkl'
    if REUSE_COMBINATION_CACHE and combination_cache_path.exists():
        import pickle
        with open(combination_cache_path, 'rb') as handle:
            combination_cache = pickle.load(handle)
    else:
        combination_cache = {comparison: build_exact_n_panel_combinations(analysis.panel_registry, comparison) for comparison in COMPARISONS}
        save_pickle(combination_cache_path, combination_cache)

    for comparison in COMPARISONS:
        eligible_targets = [target_uid for target_uid, payload in combination_cache[comparison].items() if payload.get('eligible')]
        if len(eligible_targets) != 23:
            raise RuntimeError(f'{condition}/{comparison}: expected 23 eligible proposals, found {len(eligible_targets)}')

    proposal_master_df, panel_long_df = build_review_diversity_proposal_master(analysis, combination_cache)
    if proposal_master_df.empty or panel_long_df.empty:
        raise RuntimeError(f'{condition}: review-diversity outputs are empty')

    proposal_master_df.to_csv(shared_cache_dir / 'review_diversity_proposal_master.csv', index=False)
    panel_long_df.to_csv(shared_cache_dir / 'review_diversity_panels_long.csv', index=False)

    all_ai_master_df = proposal_master_df[proposal_master_df['comparison'] == 'all_ai'].copy()
    per_model_master_df = proposal_master_df[proposal_master_df['comparison'].isin(['claude', 'gemini', 'gpt'])].copy()
    all_ai_panels_df = panel_long_df[panel_long_df['comparison'] == 'all_ai'].copy()
    per_model_panels_df = panel_long_df[panel_long_df['comparison'].isin(['claude', 'gemini', 'gpt'])].copy()

    all_ai_master_df.to_csv(tables_all_ai_dir / 'review_diversity_proposal_master.csv', index=False)
    per_model_master_df.to_csv(tables_per_model_dir / 'review_diversity_proposal_master.csv', index=False)
    all_ai_panels_df.to_csv(tables_all_ai_dir / 'review_diversity_panels_long.csv', index=False)
    per_model_panels_df.to_csv(tables_per_model_dir / 'review_diversity_panels_long.csv', index=False)

    all_tests_df = paired_review_diversity_tests(proposal_master_df, subset_label='all_proposals')
    restricted_df = proposal_master_df[proposal_master_df['target_human_n_reviews'] >= MIN_HUMAN_PANEL_FOR_SENSITIVITY].copy()
    restricted_tests_df = paired_review_diversity_tests(restricted_df, subset_label=f'human_n_gte_{MIN_HUMAN_PANEL_FOR_SENSITIVITY}')
    tests_df = pd.concat([all_tests_df, restricted_tests_df], ignore_index=True)
    tests_df.insert(0, 'condition', condition)

    all_ai_tests_df = tests_df[tests_df['comparison'] == 'all_ai'].copy()
    per_model_tests_df = tests_df[tests_df['comparison'].isin(['claude', 'gemini', 'gpt'])].copy()
    all_ai_tests_df.to_csv(tables_all_ai_dir / 'review_diversity_tests_human_vs_allai.csv', index=False)
    per_model_tests_df.to_csv(tables_per_model_dir / 'review_diversity_tests_human_vs_model.csv', index=False)

    metric_corr_df = compute_review_metric_correlation_table(proposal_master_df)
    metric_corr_df.insert(0, 'condition', condition)
    metric_corr_df.to_csv(shared_cache_dir / 'review_diversity_metric_correlations.csv', index=False)

    plot_review_diversity_paired_slopes(
        proposal_master_df,
        comparison='all_ai',
        metrics=CONFIRMATORY_METRICS,
        output_path=figures_all_ai_dir / 'paired_review_diversity_confirmatory.png',
        title=f'{condition}: Human vs All AI exact-n matched review diversity',
    )
    plot_review_diversity_effects(
        all_ai_tests_df[all_ai_tests_df['subset_label'] == 'all_proposals'],
        output_path=figures_all_ai_dir / 'review_diversity_effects_confirmatory.png',
        title=f'{condition}: Human vs All AI review-diversity effects',
        metric_class='confirmatory',
    )
    plot_review_diversity_effects(
        per_model_tests_df[per_model_tests_df['subset_label'] == 'all_proposals'],
        output_path=figures_per_model_dir / 'review_diversity_effects_confirmatory.png',
        title=f'{condition}: Human vs model review-diversity effects',
        metric_class='confirmatory',
    )
    plot_review_embedding_space(
        analysis,
        output_path=figures_all_ai_dir / 'review_space_umap.png',
        title=f'{condition}: review-space UMAP ({TEXT_VERSION})',
    )

    condition_outputs[condition] = {
        'analysis': analysis,
        'combination_cache_path': combination_cache_path,
        'proposal_master_df': proposal_master_df,
        'panel_long_df': panel_long_df,
        'tests_df': tests_df,
        'metric_corr_df': metric_corr_df,
        'tables_all_ai_dir': tables_all_ai_dir,
        'tables_per_model_dir': tables_per_model_dir,
        'figures_all_ai_dir': figures_all_ai_dir,
        'figures_per_model_dir': figures_per_model_dir,
        'shared_cache_dir': shared_cache_dir,
    }
    cross_condition_test_frames.append(tests_df)
    cross_condition_proposal_frames.append(proposal_master_df)

    print('Saved condition outputs:')
    print(f'  shared proposal master: {shared_cache_dir / "review_diversity_proposal_master.csv"}')
    print(f'  all-ai tests: {tables_all_ai_dir / "review_diversity_tests_human_vs_allai.csv"}')
    print(f'  per-model tests: {tables_per_model_dir / "review_diversity_tests_human_vs_model.csv"}')


In [ ]:
cross_tables_dir = PROJECT_ROOT / 'results' / 'tables' / 'reviews' / TEXT_VERSION / 'cross_condition'
cross_figures_dir = PROJECT_ROOT / 'results' / 'figures' / 'reviews' / TEXT_VERSION / 'cross_condition'
cross_tables_dir.mkdir(parents=True, exist_ok=True)
cross_figures_dir.mkdir(parents=True, exist_ok=True)

cross_condition_tests_df = cross_condition_summary_table(cross_condition_test_frames)
cross_condition_proposal_df = pd.concat(cross_condition_proposal_frames, ignore_index=True)
condition_contrast_df = cross_condition_ai_contrast_table(cross_condition_proposal_df)

cross_condition_tests_df.to_csv(cross_tables_dir / 'review_diversity_cross_condition_summary.csv', index=False)
condition_contrast_df.to_csv(cross_tables_dir / 'review_diversity_condition_contrasts.csv', index=False)

confirmatory_df = cross_condition_tests_df[(cross_condition_tests_df['subset_label'] == 'all_proposals') & (cross_condition_tests_df['metric_class'] == 'confirmatory')].copy()

fig, axes = plt.subplots(1, len(CONFIRMATORY_METRICS), figsize=(6 * len(CONFIRMATORY_METRICS), 5), sharey=False)
if len(CONFIRMATORY_METRICS) == 1:
    axes = [axes]
for ax, metric in zip(axes, CONFIRMATORY_METRICS):
    sub = confirmatory_df[confirmatory_df['metric'] == metric].copy()
    sns.barplot(data=sub, x='condition', y='effect_human_minus_ai_mean', hue='comparison_label', ax=ax)
    ax.axhline(0.0, color='black', linewidth=1.0)
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.set_ylabel('Mean paired difference (Human - AI)')
fig.suptitle('Cross-condition review-diversity effects (confirmatory metrics)')
fig.tight_layout()
fig.savefig(cross_figures_dir / 'review_diversity_effects_confirmatory_cross_condition.png', dpi=200, bbox_inches='tight')
plt.close(fig)

fig, axes = plt.subplots(1, len(CONFIRMATORY_METRICS), figsize=(6 * len(CONFIRMATORY_METRICS), 5), sharey=False)
if len(CONFIRMATORY_METRICS) == 1:
    axes = [axes]
for ax, metric in zip(axes, CONFIRMATORY_METRICS):
    sub = confirmatory_df[confirmatory_df['metric'] == metric].copy()
    sns.barplot(data=sub, x='condition', y='ai_to_human_ratio_mean', hue='comparison_label', ax=ax)
    ax.axhline(1.0, color='black', linewidth=1.0, linestyle='--')
    ax.set_title(metric)
    ax.set_xlabel('')
    ax.set_ylabel('AI / Human diversity-retained ratio')
fig.suptitle('Cross-condition retained-diversity ratios (confirmatory metrics)')
fig.tight_layout()
fig.savefig(cross_figures_dir / 'review_diversity_retained_ratio_confirmatory_cross_condition.png', dpi=200, bbox_inches='tight')
plt.close(fig)

display(cross_condition_tests_df.sort_values(['condition', 'comparison', 'metric', 'subset_label']).head(40))
display(condition_contrast_df.sort_values(['comparison', 'metric', 'left_condition', 'right_condition']).head(40))
